# Experiment 7.3.1 — Frozen-L2 Linear Optimization and Same-W LIF Substitution

Analysis-only notebook. It reads finalized Exp7.3.1 artifacts; no training or job submission is performed here.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    cur = start.resolve()
    for candidate in (cur, *cur.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root")

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / "notebooks" / "artifacts" / "experiment_7_3_1_frozen_l2_linear_optimization" / "frozen_l2_linear_optimization_v1"
manifest = json.loads((ROOT / "manifest.json").read_text())
runs = pd.read_csv(ROOT / "method_runs.csv")
summary = pd.read_csv(ROOT / "method_summary.csv")
contrasts = pd.read_csv(ROOT / "contrast_summary.csv")
selected_reg = pd.read_csv(ROOT / "selected_regularization.csv")
manifest


## Selected method table

Each row below is a finalized case averaged over seeds. `analog_test_ba` uses the trained Linear W; `lif_test_ba` uses the exact same W through the fixed output LIF.


In [ ]:
cols = [
    "backbone_objective", "head_objective", "case",
    "analog_test_ba_mean", "analog_test_ba_std",
    "lif_test_ba_mean", "lif_test_ba_std",
    "analog_to_lif_gap_pp_mean", "reg_lambda_mean",
]
display(summary[cols].sort_values(["backbone_objective", "head_objective", "case"]))


## Analog test BA across optimization cases


In [ ]:
case_order = manifest["cases"]
fig, ax = plt.subplots(figsize=(12, 5))
labels = []
series = []
for backbone in manifest["backbone_objectives"]:
    for objective in manifest["head_objectives"]:
        sub = summary[(summary.backbone_objective == backbone) & (summary.head_objective == objective)].set_index("case")
        vals = [sub.loc[case, "analog_test_ba_mean"] for case in case_order]
        labels.append(f"backbone={backbone}, head={objective}")
        series.append(vals)
for label, vals in zip(labels, series):
    ax.plot(case_order, vals, marker="o", label=label)
ax.set_ylabel("Test balanced accuracy")
ax.set_title("Analog Linear head")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.show()


## Same-W LIF test BA


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label, vals in zip(labels, [
    [summary[(summary.backbone_objective == backbone) & (summary.head_objective == objective)].set_index("case").loc[case, "lif_test_ba_mean"] for case in case_order]
    for backbone in manifest["backbone_objectives"]
    for objective in manifest["head_objectives"]
]):
    ax.plot(case_order, vals, marker="o", label=label)
ax.set_ylabel("Test balanced accuracy")
ax.set_title("Same trained W through fixed output LIF")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.show()


## Analog → LIF interface gap


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for backbone in manifest["backbone_objectives"]:
    for objective in manifest["head_objectives"]:
        sub = summary[(summary.backbone_objective == backbone) & (summary.head_objective == objective)].set_index("case")
        vals = [sub.loc[case, "analog_to_lif_gap_pp_mean"] for case in case_order]
        ax.plot(case_order, vals, marker="o", label=f"backbone={backbone}, head={objective}")
ax.axhline(0, linewidth=1)
ax.set_ylabel("Analog - LIF test BA (percentage points)")
ax.set_title("Same-W deployment/interface loss")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.show()


## Optimization loss and selected regularization


In [ ]:
display(
    summary[[
        "backbone_objective", "head_objective", "case",
        "train_task_ce_mean", "val_task_ce_mean",
        "reg_lambda_mean", "weight_norm_raw_mean",
    ]].sort_values(["backbone_objective", "head_objective", "case"])
)
display(selected_reg.sort_values(["backbone_objective", "head_objective", "case", "seed"]))


## Pre-registered contrasts


In [ ]:
display(contrasts.sort_values(["contrast", "backbone_objective", "head_objective"]))
